# 🚀 Project MIP — Institutional Quantitative Trading Desk & Momentum Platform
### Standalone Mobile & Cloud Engine • Python 3.10+ • C99 SIMD Acceleration

Welcome to **Project MIP (Momentum Investing Platform)** on Google Colab.
This workstation is engineered for high-performance quantitative research, multi-universe momentum scanning across all 36 official NSE indices, and blazing-fast brute force backtesting.

---

## 🛠️ Cell 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
Initializes the workstation, mounts Google Drive, and stages the database to local NVMe storage to circumvent cloud latency.

> **Storage Quota & High-Compression Note:** To prevent filling your Google Drive storage quota, `universe.db` is stored permanently in your Drive as a highly compressed gzip archive (`universe.db.gz`, ~100 MB saving over 70% storage space). When this cell runs, it streams and decompresses the archive to local high-speed NVMe storage in ~1.5s for ultra-fast <1ms queries. Any new market data fetched during scans is automatically compressed and saved back to your Google Drive.


In [ ]:
# ==============================================================================
# CELL 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
# ==============================================================================
import os, sys, platform, shutil, inspect, importlib, time
from pathlib import Path

print("🔍 PROBING CLOUD COMPUTE ENVIRONMENT...")
print(f"  • Platform:     {platform.platform()}")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • Architecture: {platform.machine()}")
print(f"  • CPU Cores:    {os.cpu_count()}")

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
WORKSPACE_DIR = Path("/content/Project_MIP_Colab") if IN_COLAB else Path.cwd()

if IN_COLAB and (WORKSPACE_DIR / ".git").exists():
    print("\n🔄 Fetching latest updates from GitHub repository...")
    !git -C {WORKSPACE_DIR} pull origin main
elif IN_COLAB and not (WORKSPACE_DIR / "core").exists():
    print("\n📦 Cloning repository into Colab workspace...")
    !git clone https://github.com/Afylx01/Project_MIP_Colab.git {WORKSPACE_DIR}

if WORKSPACE_DIR.exists():
    os.chdir(WORKSPACE_DIR)
    if str(WORKSPACE_DIR) not in sys.path:
        sys.path.insert(0, str(WORKSPACE_DIR))

# Purge any stale in-memory cached modules from previous cell runs or pre-pull state
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("scripts", "core", "brute_force", "nextgen")):
        sys.modules.pop(mod_name, None)

!pip install -q -r requirements.txt
print(f"\n✓ Workspace active at: {WORKSPACE_DIR}")

# Automatically mount Google Drive if running in Colab
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    print("\n📁 Mounting Google Drive...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")

# Bulletproof module loading with reload & inline fallback
try:
    import scripts.cloud_sync
    importlib.reload(scripts.cloud_sync)
    from scripts.cloud_sync import (
        sync_universe_from_gdrive,
        benchmark_storage_latency,
        is_gzipped_file,
        decompress_file,
        PUBLIC_UNIVERSE_DB_FILE_ID,
        PUBLIC_UNIVERSE_DB_URL,
    )
except Exception:
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith(("scripts", "core", "brute_force", "nextgen")):
            sys.modules.pop(mod_name, None)
    from scripts.cloud_sync import (
        sync_universe_from_gdrive,
        benchmark_storage_latency,
        is_gzipped_file,
        decompress_file,
        PUBLIC_UNIVERSE_DB_FILE_ID,
        PUBLIC_UNIVERSE_DB_URL,
    )

if 'is_gzipped_file' not in globals() or not callable(globals().get('is_gzipped_file')):
    def is_gzipped_file(file_path):
        p = Path(file_path)
        if not p.exists() or p.stat().st_size < 2:
            return False
        try:
            with open(p, "rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except Exception:
            return False

if 'decompress_file' not in globals() or not callable(globals().get('decompress_file')):
    def decompress_file(src_gz, dest_raw):
        import gzip
        src, dest = Path(src_gz), Path(dest_raw)
        tmp = dest.parent / f"{dest.name}.tmp_decomp_{int(time.time())}"
        with gzip.open(src, "rb") as f_in, open(tmp, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        tmp.replace(dest)
        return src.stat().st_size, dest.stat().st_size

from core.data_engine import get_db_path, get_universe_statistics

print("\n⚡ Staging universe.db to local NVMe storage (with gzip decompress)...")
ok, status, meta = sync_universe_from_gdrive(verbose=True)

db_target = get_db_path()
if not db_target.exists() and IN_COLAB:
    print(f"\n📥 Downloading universe.db directly via Google Drive File ID ({PUBLIC_UNIVERSE_DB_FILE_ID})...")
    !gdown --id {PUBLIC_UNIVERSE_DB_FILE_ID} -O {db_target}
    if db_target.exists() and is_gzipped_file(db_target):
        print("  • Decompressing downloaded gzip archive to SQLite...")
        decompress_file(db_target, db_target)

if db_target.exists():
    stats = get_universe_statistics()
    print("")
    print("✓ DATABASE INTEGRITY CONFIRMED:")
    print(f"  • Total Rows:     {stats['total_rows']:,}")
    print(f"  • Unique Symbols: {stats['unique_symbols']}")
    print(f"  • Date Span:      {stats['min_date']} → {stats['max_date']}")
    print(f"  • Local NVMe Size:{stats['file_size_mb']:.2f} MB")
    print("")
    print("⚡ Benchmarking Storage Latency...")
    bm = benchmark_storage_latency(db_target, num_queries=20)
    print(f"  • Average Latency: {bm['avg_latency_ms']:.2f} ms ({'NVMe Ultra-Fast' if bm['is_nvme_grade'] else 'Network-Bound'})")
else:
    print("")
    print("⚠ universe.db not found on Google Drive.")
    print("ℹ Direct Download Link:")
    print(f"  {PUBLIC_UNIVERSE_DB_URL}")
    print("  Or run: !gdown --id 1mnaMfODbDwcdyG5NyQrLCunWKwYir1aT -O data/universe.db")

## 📊 Cell 2: Interactive Cockpit & Momentum Screener Dashboard
Interactive UI powered by **IPyWidgets** and **Matplotlib**.
Guarantees scan execution on the **latest actual completed trading day of EOD**, with real-time Telegram dispatch, Excel generation, browser auto-download, and database synchronization audits.


In [ ]:
# ==============================================================================
# CELL 2: Interactive IPyWidgets UI Dashboard & Tearsheet Generator
# ==============================================================================
import sys, importlib, inspect
for mod_name in ["core.scanner", "scripts.scanner_hardening", "core.telegram_sender", "core.auto_fetch", "core.settings"]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from pathlib import Path
from core.scanner import format_screener_tearsheet
from scripts.scanner_hardening import HardenedScanner, NIFTY_INDICES_CATALOG, trigger_colab_download
from core.data_engine import get_expected_latest_eod, get_latest_date

u_options = ["NIFTY750", "NIFTY500", "NIFTY50", "NIFTYBANK", "NIFTYIT", "NIFTYAUTO", "NIFTYPHARMA", "ALL"]
w_universe = widgets.Dropdown(options=u_options, value="NIFTY750", description="Universe:")
w_top_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Top N:")
w_allow_stale = widgets.Checkbox(value=False, description="Allow Stale Data")
w_export_excel = widgets.Checkbox(value=True, description="Generate Excel 📊")
w_send_telegram = widgets.Checkbox(value=True, description="Send to Telegram 📱")
w_download_excel = widgets.Checkbox(value=True, description="Auto-Download Excel 📥")
w_run_btn = widgets.Button(description="🚀 Run Momentum Scan", button_style="success", icon="play")
w_output = widgets.Output()

expected_session = get_expected_latest_eod()
db_latest = get_latest_date()
print(f"Market State: Latest Expected EOD: {expected_session} | Database Latest: {db_latest}")

_scan_is_running = False

def on_run_clicked(b):
    global _scan_is_running
    if _scan_is_running:
        return
    _scan_is_running = True
    w_run_btn.disabled = True
    orig_desc = w_run_btn.description
    orig_style = w_run_btn.button_style
    w_run_btn.description = "⏳ Scanning in progress..."
    w_run_btn.button_style = "warning"

    try:
        with w_output:
            clear_output(wait=True)
            print(f"Starting scan on {w_universe.value} (Top {w_top_n.value})...")
            scanner = HardenedScanner(WORKSPACE_DIR)
            
            sig = inspect.signature(scanner.run_scan)
            scan_kwargs = {
                "as_of_date": None,
                "universe_mode": w_universe.value,
                "top_n": w_top_n.value,
                "auto_update_universe": True,
                "export_csv": True,
                "export_excel": w_export_excel.value,
                "allow_stale_data": w_allow_stale.value,
            }
            if "send_telegram" in sig.parameters:
                scan_kwargs["send_telegram"] = w_send_telegram.value
            if "auto_download_excel" in sig.parameters:
                # Handled cleanly in UI layer to avoid duplicate browser downloads
                scan_kwargs["auto_download_excel"] = False

            top_df, breadth, sectors, regime, top_etf, deliv = scanner.run_scan(**scan_kwargs)
            scanned_date = top_df.attrs.get("scanned_date") or db_latest
            print(format_screener_tearsheet(top_df, breadth, sectors, regime, scanned_date))
            
            # Database Synchronization Status Banner
            db_audit = top_df.attrs.get("db_audit")
            if db_audit:
                print("\n" + "=" * 76)
                if db_audit.get("is_updated"):
                    print("💾 DATABASE AUDIT: 🟢 UPDATED WITH NEW MARKET SESSIONS!")
                    print(f"  • New Rows Added:  +{db_audit.get('new_rows_added', 0):,} rows committed to SQLite")
                    print(f"  • Date Transition: {db_audit.get('pre_latest_date')} ➔ {db_audit.get('post_latest_date')}")
                    print(f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table")
                    if db_audit.get("gdrive_backed_up"):
                        print("  • Google Drive:    ✓ Backed up to /MyDrive/MIP/universe.db.gz")
                else:
                    print("💾 DATABASE AUDIT: 🔵 ALREADY UP-TO-DATE (No new sessions available)")
                    print(f"  • Latest Session:  {db_audit.get('post_latest_date')}")
                    print(f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table")
                print("=" * 76)
            
            # Plot top candidates FIRST before triggering browser download
            if top_df is not None and not top_df.empty and "volar_score" in top_df.columns:
                plt.figure(figsize=(10, 4))
                plot_data = top_df.head(10).sort_values("volar_score", ascending=True)
                plt.barh(plot_data["symbol"], plot_data["volar_score"], color="#00adb5")
                plt.title(f"Top 10 Momentum Candidates [{w_universe.value}] ({scanned_date})")
                plt.xlabel("Volar Momentum Score")
                plt.tight_layout()
                plt.show()

            # Single, Deduped Automatic Browser Download
            excel_path = top_df.attrs.get("excel_path")
            if excel_path and Path(excel_path).exists():
                print("")
                print(f"✓ Institutional Excel Workbook ready: {excel_path}")
                if w_download_excel.value:
                    try:
                        print(f"📥 Triggering browser download for {Path(excel_path).name}...")
                        dl_ok = trigger_colab_download(excel_path, cooldown_seconds=20.0)
                        if dl_ok:
                            print("✓ Browser download initiated.")
                        else:
                            print("✓ Download already completed (deduplication active).")
                    except Exception as e_dl:
                        print(f"ℹ️ (File ready at: {excel_path} - Note: {e_dl})")
    except Exception as e:
        with w_output:
            print(f"❌ Scan Error: {e}")
    finally:
        w_run_btn.disabled = False
        w_run_btn.description = orig_desc
        w_run_btn.button_style = orig_style
        _scan_is_running = False

try:
    w_run_btn.on_click(on_run_clicked, remove=True)
except Exception:
    pass
w_run_btn.on_click(on_run_clicked)

display(widgets.VBox([
    widgets.HBox([w_universe, w_top_n]),
    widgets.HBox([w_allow_stale, w_export_excel, w_send_telegram, w_download_excel]),
    w_run_btn,
    w_output
]))


## ⚡ Cell 3: High-Performance C99 SIMD Backtesting & Brute Force Cockpit
Execute custom-date multi-factor backtests or launch parallel multi-threaded brute-force parameter optimization sweeps in C99 SIMD (AVX2 / NEON). Features interactive charts, tearsheet metrics, and automated CSV result downloads.


In [ ]:
# ==============================================================================
# CELL 3: Multi-Architecture C99 SIMD Backtesting & Brute Force Cockpit
# ==============================================================================
import os, sys, time, inspect
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from brute_force.bridge import compile_native_kernel
from brute_force.runner import BruteForceCampaignRunner

# 1. Compile native C99 SIMD kernel (AVX2/NEON)
so_path = compile_native_kernel(force_recompile=False)
print(f"✓ Native SIMD Kernel compiled: {so_path.name} ({so_path.stat().st_size:,} bytes)")

# 2. Maintain singleton runner instance to reuse cached TensorCube in RAM
if "_bf_runner" not in globals() or _bf_runner is None:
    _bf_runner = BruteForceCampaignRunner()
runner = _bf_runner

# 3. Interactive UI Controls
w_engine_mode = widgets.RadioButtons(
    options=["🎯 Custom Date & Parameter Backtest", "⚡ Parallel Brute Force Optimization Sweep"],
    value="🎯 Custom Date & Parameter Backtest",
    description="Engine Mode:",
    layout={"width": "max-content"}
)

# Mode 1 Controls: Custom Date Backtest
era_presets = {
    "Full 10-Year (2016–2026)": ("2016-01-01", "2026-08-31"),
    "Post-COVID Bull (2020–2024)": ("2020-04-01", "2024-12-31"),
    "COVID Crash & Recovery (2020)": ("2020-01-01", "2020-09-30"),
    "Recent 2-Year (2024–2026)": ("2024-01-01", "2026-10-14"),
    "Mid-Cap Bear (2018–2019)": ("2018-01-01", "2019-12-31"),
    "Custom Dates": ("", ""),
}

w_bt_preset = widgets.Dropdown(options=list(era_presets.keys()), value="Full 10-Year (2016–2026)", description="Era Preset:")
w_bt_start = widgets.Text(value="2016-01-01", description="Start Date:")
w_bt_end = widgets.Text(value="2026-08-31", description="End Date:")
w_bt_universe = widgets.Dropdown(options=["NIFTY750", "NIFTY500", "ALL"], value="NIFTY750", description="Universe:")
w_bt_schedule = widgets.Dropdown(options=["monthly", "weekly", "quarterly"], value="monthly", description="Schedule:")
w_bt_top_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Top N:")
w_bt_lookback = widgets.Dropdown(options=[126, 189, 252, 315], value=252, description="Lookback:")
w_bt_ema = widgets.Dropdown(options=[100, 150, 200, 250], value=200, description="EMA Trend:")
w_bt_retrace = widgets.FloatSlider(value=0.80, min=0.60, max=0.95, step=0.05, description="Retrace:")
w_bt_sector_cap = widgets.IntSlider(value=2, min=1, max=5, description="Max/Sector:")
w_bt_sizing = widgets.Dropdown(options=["atr_risk_parity", "equal_weight"], value="atr_risk_parity", description="Sizing Mode:")

w_alpha_res = widgets.Checkbox(value=False, description="Residual Momentum (Blitz 2011)")
w_alpha_vol = widgets.Checkbox(value=False, description="EWMA Vol Targeting (Harvey 2018)")
w_alpha_flow = widgets.Checkbox(value=False, description="Delivery Flow Absorption Veto")

w_btn_run_bt = widgets.Button(description="🚀 Run Custom Backtest", button_style="primary", icon="play")

# Mode 2 Controls: Brute Force Parameter Sweep
sweep_presets = [
    "Tier 1: Lookback & EMA Trend Grid (35 runs)",
    "Tier 2: Portfolio Size & Exit Buffer (44 runs)",
    "Tier 3: Full Neighborhood Grid (810 runs)",
    "Custom Parameter Sweep",
]
w_sw_preset = widgets.Dropdown(options=sweep_presets, value="Tier 1: Lookback & EMA Trend Grid (35 runs)", description="Sweep Type:")
w_sw_universe = widgets.Dropdown(options=["NIFTY750", "NIFTY500"], value="NIFTY750", description="Universe:")
w_sw_schedule = widgets.Dropdown(options=["monthly", "weekly", "quarterly"], value="monthly", description="Schedule:")
w_sw_start = widgets.Text(value="2016-01-01", description="Start Date:")
w_sw_end = widgets.Text(value="2026-08-31", description="End Date:")
w_sw_threads = widgets.IntSlider(value=min(8, os.cpu_count() or 4), min=1, max=16, description="Threads:")
w_sw_auto_dl = widgets.Checkbox(value=True, description="Auto-Download CSV 📥")
w_btn_run_sw = widgets.Button(description="⚡ Launch Parallel C99 Sweep", button_style="warning", icon="bolt")

w_output_bt = widgets.Output()

def on_preset_changed(change):
    if change["new"] in era_presets and change["new"] != "Custom Dates":
        s, e = era_presets[change["new"]]
        w_bt_start.value = s
        w_bt_end.value = e

w_bt_preset.observe(on_preset_changed, names="value")

box_backtest = widgets.VBox([
    widgets.HBox([w_bt_preset, w_bt_universe]),
    widgets.HBox([w_bt_start, w_bt_end]),
    widgets.HBox([w_bt_schedule, w_bt_top_n]),
    widgets.HBox([w_bt_lookback, w_bt_ema]),
    widgets.HBox([w_bt_retrace, w_bt_sector_cap]),
    widgets.HBox([w_bt_sizing]),
    widgets.HBox([w_alpha_res, w_alpha_vol, w_alpha_flow]),
    w_btn_run_bt
])

box_sweep = widgets.VBox([
    widgets.HBox([w_sw_preset, w_sw_universe]),
    widgets.HBox([w_sw_schedule, w_sw_threads]),
    widgets.HBox([w_sw_start, w_sw_end]),
    widgets.HBox([w_sw_auto_dl]),
    w_btn_run_sw
])

ui_container = widgets.VBox([box_backtest])

def on_mode_changed(change):
    if "Custom" in change["new"]:
        ui_container.children = [box_backtest]
    else:
        ui_container.children = [box_sweep]

w_engine_mode.observe(on_mode_changed, names="value")

_bt_is_running = False

def handle_run_backtest(b):
    global _bt_is_running
    if _bt_is_running:
        return
    _bt_is_running = True
    w_btn_run_bt.disabled = True
    _orig_bt_desc = w_btn_run_bt.description
    w_btn_run_bt.description = "⏳ Simulating..."
    with w_output_bt:
        clear_output()
        print(f"🚀 Initializing C99 Custom Backtest [{w_bt_universe.value}]...")
        print(f"  • Date Span:  {w_bt_start.value} ➔ {w_bt_end.value} ({w_bt_schedule.value})")
        print(f"  • Parameters: Lookback={w_bt_lookback.value}d | EMA={w_bt_ema.value}d | Top {w_bt_top_n.value} | Retrace={w_bt_retrace.value:.2f} | Max/Sec={w_bt_sector_cap.value}")
        print(f"  • Sizing:     {w_bt_sizing.value} | Alphas: ResMom={w_alpha_res.value}, VolTarget={w_alpha_vol.value}, Delivery={w_alpha_flow.value}")
        t0 = time.perf_counter()
        try:
            res = runner.run_custom_backtest(
                start_date=w_bt_start.value,
                end_date=w_bt_end.value or None,
                universe_name=w_bt_universe.value,
                schedule=w_bt_schedule.value,
                lookback=w_bt_lookback.value,
                ema_trend=w_bt_ema.value,
                top_n=w_bt_top_n.value,
                retrace_threshold=w_bt_retrace.value,
                max_per_sector=w_bt_sector_cap.value,
                sizing_mode=w_bt_sizing.value,
                use_residual_mom=w_alpha_res.value,
                use_vol_targeting=w_alpha_vol.value,
                use_delivery_veto=w_alpha_flow.value
            )
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
            
            print("\n" + "=" * 76)
            print(f" PROJECT MIP: INSTITUTIONAL PERFORMANCE TEARSHEET ({res['start_date']} ➔ {res['end_date']})")
            print("=" * 76)
            print(f"  • Initial Capital:    ₹{res['initial_capital']:,.2f}")
            print(f"  • Ending Capital:     ₹{res['ending_equity']:,.2f} (Total Return: {((res['ending_equity']/res['initial_capital'])-1)*100:+.2f}%)")
            print(f"  • Annualized CAGR:    {res['cagr_pct']:+.2f}%")
            print(f"  • Sharpe Ratio:       {res['sharpe_ratio']:.2f}")
            print(f"  • Sortino Ratio:      {res['sortino_ratio']:.2f}")
            print(f"  • Calmar Ratio:       {res['calmar_ratio']:.2f}")
            print(f"  • Maximum Drawdown:   {res['max_drawdown_pct']:.2f}%")
            print(f"  • Win Rate:           {res['win_rate_pct']:.1f}% ({res['winning_trades']} wins / {res['total_trades']} trades)")
            print(f"  • Annual Turnover:    {res['turnover_annual_pct']:.1f}%")
            print(f"  • C99 Kernel Latency: {res['sim_latency_ms']:.2f} ms (Total compute: {elapsed_ms:.1f} ms)")
            print("=" * 76)
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), gridspec_kw={"height_ratios": [2.5, 1.2]}, sharex=True)
            dates = pd.to_datetime(res["rebalance_dates"])
            eq = np.array(res["equity_curve"])
            bench = np.array(res["benchmark_equity"])
            dd = np.array(res["drawdown_pct_series"])
            
            ax1.plot(dates, eq, label=f"MIP Strategy ({res['universe']}) [CAGR: {res['cagr_pct']:+.1f}%]", color="#00adb5", lw=2.2)
            ax1.plot(dates, bench, label="NIFTY 500 Benchmark", color="#ff79c6", lw=1.6, ls="--", alpha=0.85)
            ax1.set_title(f"MIP Institutional Backtest • {res['universe']} ({res['start_date']} to {res['end_date']})", fontsize=13, fontweight="bold")
            ax1.set_ylabel("Portfolio Value (₹)", fontsize=11)
            ax1.grid(True, alpha=0.25, ls=":")
            ax1.legend(loc="upper left", frameon=True)
            ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"₹{x:,.0f}"))
            
            ax2.fill_between(dates, dd, 0, color="#ff5555", alpha=0.4, label=f"Underwater Drawdown (Max: {res['max_drawdown_pct']:.1f}%)")
            ax2.plot(dates, dd, color="#ff5555", lw=1.2)
            ax2.set_ylabel("Drawdown %", fontsize=11)
            ax2.set_xlabel("Rebalance Session Date", fontsize=11)
            ax2.grid(True, alpha=0.25, ls=":")
            ax2.legend(loc="lower left", frameon=True)
            ax2.set_ylim([min(-35.0, np.min(dd) * 1.15), 2.0])
            
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"❌ Backtest Error: {e}")
        finally:
            w_btn_run_bt.disabled = False
            w_btn_run_bt.description = _orig_bt_desc
            _bt_is_running = False

_sw_is_running = False

def handle_run_sweep(b):
    global _sw_is_running
    if _sw_is_running:
        return
    _sw_is_running = True
    w_btn_run_sw.disabled = True
    _orig_sw_desc = w_btn_run_sw.description
    w_btn_run_sw.description = "⏳ Sweeping..."
    with w_output_bt:
        clear_output()
        print(f"⚡ Launching Parallel C99 SIMD Optimization Sweep ({w_sw_universe.value})...")
        print(f"  • Sweep Selection: {w_sw_preset.value}")
        print(f"  • Date Span:       {w_sw_start.value} ➔ {w_sw_end.value} ({w_sw_schedule.value})")
        print(f"  • Concurrency:     {w_sw_threads.value} CPU Worker Threads")
        t0 = time.perf_counter()
        try:
            if "Tier 1" in w_sw_preset.value:
                df = runner.run_grid_preset(preset="tier1", universe_name=w_sw_universe.value, num_threads=w_sw_threads.value)
            elif "Tier 2" in w_sw_preset.value:
                df = runner.run_grid_preset(preset="tier2", universe_name=w_sw_universe.value, num_threads=w_sw_threads.value)
            elif "Tier 3" in w_sw_preset.value:
                df = runner.run_grid_preset(preset="tier3", universe_name=w_sw_universe.value, num_threads=w_sw_threads.value)
            else:
                df = runner.run_custom_sweep(
                    start_date=w_sw_start.value,
                    end_date=w_sw_end.value or None,
                    universe_name=w_sw_universe.value,
                    schedules=[w_sw_schedule.value],
                    num_threads=w_sw_threads.value
                )
            
            elapsed_s = time.perf_counter() - t0
            sims_per_sec = len(df) / max(0.001, elapsed_s)
            
            print(f"\n✓ Optimization Sweep Complete: {len(df):,} parameter sets evaluated in {elapsed_s:.2f}s ({sims_per_sec:,.0f} sims/sec)")
            print("\n🏆 TOP 10 OPTIMAL PARAMETER CONFIGURATIONS (RANKED BY SHARPE):")
            top10 = df.head(10).copy()
            cols_to_show = [c for c in ["param_id", "lookback", "ema_trend", "top_n", "retrace_threshold", "cagr", "sharpe", "sortino", "max_drawdown", "win_rate", "ending_equity"] if c in top10.columns]
            print(top10[cols_to_show].to_string(index=False))
            
            out_dir = Path("reports/brute_force")
            out_dir.mkdir(parents=True, exist_ok=True)
            csv_path = out_dir / "brute_force_sweep_results.csv"
            df.to_csv(csv_path, index=False)
            print(f"\n📁 Full sweep dataset exported to: {csv_path} ({len(df)} rows)")
            
            plt.figure(figsize=(10, 4))
            top_plot = top10.sort_values("sharpe", ascending=True)
            labels = [f"LB={int(r['lookback'])}d EMA={int(r['ema_trend'])}d N={int(r['top_n'])}" for _, r in top_plot.iterrows()]
            plt.barh(labels, top_plot["sharpe"], color="#00adb5")
            plt.title("Top 10 Configurations — Sharpe Ratio Distribution", fontsize=12, fontweight="bold")
            plt.xlabel("Annualized Sharpe Ratio")
            plt.tight_layout()
            plt.show()
            
            if w_sw_auto_dl.value:
                try:
                    from scripts.scanner_hardening import trigger_colab_download
                    print(f"📥 Triggering browser download for {csv_path.name}...")
                    dl_ok = trigger_colab_download(csv_path, cooldown_seconds=20.0)
                    if dl_ok:
                        print("✓ Browser download initiated.")
                    else:
                        print("✓ Download already completed (deduplication active).")
                except Exception as e_dl:
                    print(f"ℹ️ (File ready on disk at: {csv_path})")
        except Exception as e:
            print(f"❌ Sweep Error: {e}")
        finally:
            w_btn_run_sw.disabled = False
            w_btn_run_sw.description = _orig_sw_desc
            _sw_is_running = False

try:
    w_btn_run_bt.on_click(handle_run_backtest, remove=True)
    w_btn_run_sw.on_click(handle_run_sweep, remove=True)
except Exception:
    pass
w_btn_run_bt.on_click(handle_run_backtest)
w_btn_run_sw.on_click(handle_run_sweep)

display(widgets.VBox([
    w_engine_mode,
    ui_container,
    w_output_bt
]))


## 🖥️ Cell 4: Headless CLI Execution & Batch Commands
Run any of the 18 CLI functions directly from the notebook.


In [ ]:
# ==============================================================================
# CELL 4: Headless Terminal Execution Examples
# ==============================================================================
# 1. Run live momentum scan on NIFTY 750 (Total Market)
!python main.py --option 1 --universe NIFTY750 --allow-stale

# 2. Run blazing fast brute force baseline simulation
# !python main.py --brute-force --brute-force-mode baseline

# 3. Run complete 4-tier optimization matrix campaign
# !python scripts/run_optimization_matrix.py